# Phụ lục B — Đối chiếu Đặc tả Đồ án và Giới hạn Thực nghiệm

Ma trận kiểm định dưới đây đối chiếu toàn bộ yêu cầu kỹ thuật của **Milestone 1**, **Milestone 2**, tiêu chí **Jupyter Book** và hướng dẫn **QA của giảng viên** trên quy mô trọn vẹn **25.000 bài báo**:

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

BOOK_PALETTE = {
    "navy": "#0F4C81",
    "teal": "#0D9488",
    "emerald": "#059669",
    "amber": "#D97706",
    "crimson": "#B31B1B",
    "slate": "#64748B",
    "sky": "#0284C7",
    "grid": "#CBD5E1",
}
MEMBER_COLORS = {
    "DAT": "#0F4C81",
    "TUAN": "#0D9488",
    "TAN": "#D97706",
    "TRI": "#059669",
    "TRIET": "#B31B1B",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "figure.figsize": (9.8, 4.3),
    "figure.dpi": 180,
    "savefig.dpi": 180,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 11.0,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#0F4C81",
    "axes.labelsize": 9.5,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.9,
    "ytick.labelsize": 8.9,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": "--",
    "grid.alpha": 0.42,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "—"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    bar_cols: list[str] | dict[str, tuple[str, float | None]] | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    bar_color: str = "var(--ah-bar-fill)",
    vmax: float | None = None,
):
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    for c in int_set:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round().astype("Int64")

    fmt_map: dict[str, object] = {}
    for col in out.columns:
        if col in custom_fmt:
            fmt_map[col] = custom_fmt[col]
        elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
            fmt_map[col] = fmt_int
        elif col in pct_set:
            fmt_map[col] = lambda v, d=precision, s=pct_scale100: fmt_pct(v, d, s)
        elif pd.api.types.is_float_dtype(out[col]):
            fmt_map[col] = lambda v, d=precision: fmt_num(v, d)

    styler = out.style.hide(axis="index").set_table_attributes('class="dataframe ah-styled-table"')
    if caption:
        styler = styler.set_caption(caption)
    if fmt_map:
        styler = styler.format(fmt_map)

    if isinstance(bar_cols, dict):
        bar_spec = bar_cols
    else:
        bar_spec = {c: (bar_color, vmax) for c in (bar_cols or [])}

    for bc, (bcol, bmax) in bar_spec.items():
        if bc in out.columns:
            num_s = pd.to_numeric(out[bc], errors="coerce")
            hi = float(bmax) if bmax is not None else float(num_s.max() or 1.0)
            styler = styler.bar(subset=[bc], color=bcol, vmin=0.0, vmax=max(hi, 1e-9))

    num_cols = [
        c for c in out.columns
        if c in int_set or c in pct_set or c in bar_spec or pd.api.types.is_numeric_dtype(out[c])
    ]
    if num_cols:
        styler = styler.set_properties(
            subset=num_cols,
            **{"text-align": "right", "font-variant-numeric": "tabular-nums"},
        )
    return styler


print(
    f"Toàn nhóm (5 thành viên): {fmt_int(scope['papers'])} bài báo / "
    f"{fmt_int(scope['versions'])} phiên bản | Cập nhật: {data['report_date']}"
)

In [ ]:
sa = m1["source_audit"]
rows = [
    ("M1", "Discovery đủ dải 5 thành viên & OAI-PMH 2025", "Đạt 25.000 / 25.000 ID (0 trùng chéo)", "Tích hợp oaipmh.arxiv.org + --resume trên Colab CPU cho phân vùng TRI (24120470)"),
    ("M1", "Tải nguồn đa phiên bản, lọc ảnh & kiểm toán", f"{fmt_int(m1['source_verified'])} / {fmt_int(scope['versions'])} phiên bản xác minh", f"Giảm 89,30% dung lượng (bytes); còn 11 phiên bản chờ từ TRIET + {fmt_int(sa.get('not_found_verified', 0))} HTTP 404"),
    ("M1", "Trích dẫn S2 batch/phân trang & course export", f"{fmt_int(m1['verified_reference_papers'])} / {fmt_int(scope['papers'])} bài S2 xác minh", f"{fmt_int(m1['working_reference_rows'])} working rows (0 mồ côi) / {fmt_int(m1['arxiv_reference_rows'])} cạnh đích arXiv"),
    ("M1", "Đủ 7 chỉ số hiệu năng Crawler trên Colab CPU", "Đạt trọn vẹn 7/7 chỉ số bắt buộc", "TB 8,06s/bài (4 full) / 8,35s/bài (đủ 25k); Peak RSS 1.227,73 MiB; Discovery 2.094,9s"),
    ("M2", "Multi-file gathering & phân giải đa root", "Đạt (cứu 1.706 phiên bản đa root nhờ tiêu đề)", "26 phiên bản thiếu \\documentclass; 99 phiên bản dùng macro động trong \\input -> partial"),
    ("M2", "Cây phân cấp (sentence/math/figure/list)", "97,34% phiên bản TeX dựng được cây hợp lệ", "791 phiên bản (9.087 lượt cảnh báo) có macro \\item tùy biến ngoài list chuẩn -> giữ ở partial"),
    ("M2", "Chuẩn hóa LaTeX/Toán & khử trùng 2 tầng", f"Elements -{fmt_pct(m2['deduplication']['elements']['reduction_rate'], 2, True)}; BibTeX -{fmt_pct(m2['deduplication']['bibtex']['reduction_rate'], 2, True)}", "Đạt -46,95% / -50,41% trên bài đa phiên bản; giữ .bbl & đổi khóa \\cite{} xuyên version"),
    ("M2", "Kiểm chuẩn 5 bảng ngoài bộ nhớ (SQLite DAG)", "structural_valid = True (0 lỗi)", f"Elements {fmt_int(m2['rows']['elements'])}; Hierarchy {fmt_int(m2['rows']['hierarchy'])}; BibTeX {fmt_int(m2['rows']['bibtex'])}"),
    ("M2", "Benchmark 5 phiên lạnh trên Colab CPU-only", "Đạt (5/5 phiên độc lập, reused = 0)", f"{fmt_num(bench['totals']['total_profiler_sec'], 2)}s ({fmt_num(bench['totals']['avg_sec_per_unique_publication'], 4)}s/bài); Peak RSS {fmt_num(bench['totals']['max_session_peak_rss_mib'], 2)} MiB"),
    ("Book", "Jupyter Book động, tương tác & Tiếng Việt", "Biên dịch --all -W đạt 0 cảnh báo", "18 tổ hợp hình 180-DPI + 29 bảng Styler có Click-to-Sort & lọc nhanh; giữ URL cố định"),
]
display(style_table(
    pd.DataFrame(rows, columns=["Mốc", "Hạng mục yêu cầu", "Kết quả xác minh (25.000 bài)", "Ghi chú kỹ thuật & giới hạn thực nghiệm"]),
    caption="Bảng B.1. Ma trận đối chiếu toàn diện yêu cầu đồ án và giới hạn thực nghiệm.",
))

## Tài liệu và mã nguồn tham chiếu

- [Đặc tả chính thức Milestone 1 — Data Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [Đặc tả chính thức Milestone 2 — Hierarchical Parsing & Standardization](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) & [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`README.md`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/README.md).
- Mã nguồn triển khai: [`src/parser/validation.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/validation.py).